# M4-02-04 scalar observer — Git / Colab L4
Git supplies code and durable evidence. No ZIP or Drive requirement.
The committed template requires exact CODE_SHA; the handoff copy fills it after push.
Only K=0 then K=2, actual worker B=1. No full grid or profitability conclusions.
Run cells sequentially. Smoke failures still flow to the evidence retention cell.


In [ ]:
from google.colab import userdata
from pathlib import Path
from datetime import datetime, timezone
import json, os, platform, re, shutil, subprocess, sys, tempfile, uuid
REPO_URL = "https://github.com/theflyingkhui04/do-an-ptit-d22.git"
BRANCH = "feat/m4-02-04-scheduler-telemetry-v02"
CODE_SHA = "SET_EXACT_PUSHED_SHA"
RUN_REPO = Path("/content/do-an-ptit-d22-run")
EVIDENCE_REPO = Path("/content/do-an-ptit-d22-evidence")
if CODE_SHA == "SET_EXACT_PUSHED_SHA":
    CODE_SHA = input("Paste the exact pushed CODE_SHA from the handoff: ").strip()
assert re.fullmatch(r"[0-9a-f]{40}", CODE_SHA), "Exact CODE_SHA required; no branch/HEAD fallback"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-scalar-observer-l4-" + uuid.uuid4().hex[:8]
RUN_REL = "experiments/runs/m4_02_04/" + RUN_ID
RUN_DIR = RUN_REPO / RUN_REL
assert platform.system() == "Linux", "CANONICAL ENVIRONMENT MISMATCH: Linux required"
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=20)
query = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"], capture_output=True, text=True, timeout=20)
rows = query.stdout.strip().splitlines()
assert smi.returncode == query.returncode == 0 and len(rows) == 1 and rows[0].split(",")[0].strip() == "NVIDIA L4", "CANONICAL ENVIRONMENT MISMATCH: exact NVIDIA L4 required"
import torch  # Inspection only; models use the isolated process below.
assert torch.cuda.is_available() and torch.cuda.get_device_name(0) == "NVIDIA L4" and torch.cuda.get_device_capability(0) == (8, 9), "CANONICAL ENVIRONMENT MISMATCH: L4/sm89"
assert int(rows[0].split(",")[1].strip().split(".")[0]) >= 580, "CANONICAL ENVIRONMENT MISMATCH: CUDA 13 driver support"
print("CODE_SHA:", CODE_SHA)
print("GPU:", query.stdout.strip(), "SM:", torch.cuda.get_device_capability(0))


In [ ]:
token = userdata.get("GITHUB_TOKEN")
assert token, "Missing Colab Secret: GITHUB_TOKEN"
assert not RUN_REPO.exists() and not EVIDENCE_REPO.exists(), "Existing checkouts; preserve, do not reset/rebase"
with tempfile.TemporaryDirectory(prefix="m40204-clone-auth-") as auth_dir:
    askpass = Path(auth_dir) / "askpass.py"
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['M40204_GITHUB_TOKEN'])\n".replace("\\n", "\n"))
    askpass.chmod(0o700)
    auth_env = dict(os.environ, M40204_GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="credential.helper", GIT_CONFIG_VALUE_0="", GIT_TRACE="0", GIT_TRACE_CURL="0", GIT_CURL_VERBOSE="0")
    try:
        for checkout in (RUN_REPO, EVIDENCE_REPO):
            subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(checkout)], check=True, env=auth_env)
    finally:
        auth_env.pop("M40204_GITHUB_TOKEN", None)
        del token
subprocess.run(["git", "checkout", "--detach", CODE_SHA], cwd=RUN_REPO, check=True)
run_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=RUN_REPO, text=True).strip()
evidence_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=EVIDENCE_REPO, text=True).strip()
assert run_sha == evidence_sha == CODE_SHA, "Initial checkout SHA mismatch; no automatic reconciliation"
sys.path.insert(0, str(RUN_REPO))
from experiments.scalar_colab_git import source_gate, github_auth, push_evidence
source_gate(RUN_REPO, CODE_SHA)
RUN_DIR.mkdir(parents=True, exist_ok=False)
SETUP = RUN_DIR / "setup"
SETUP.mkdir()
(SETUP / "nvidia-smi.txt").write_text(smi.stdout + smi.stderr)
(SETUP / "source_gate_initial.json").write_text(json.dumps(source_gate(RUN_REPO, CODE_SHA, RUN_REL), indent=2))
RUN_ENV = dict(os.environ, PYTHONPATH=str(RUN_REPO), VLLM_USE_V2_MODEL_RUNNER="0", VLLM_ENABLE_V1_MULTIPROCESSING="0", PTIT_SCALAR_CODE_SHA=CODE_SHA, PTIT_SCALAR_RUNTIME_REL=RUN_REL)
RUN_ENV.pop("GITHUB_TOKEN", None)
RUN_ENV.pop("M40204_GITHUB_TOKEN", None)
print("TWO CHECKOUTS / DETACHED SOURCE GATE: PASS", CODE_SHA)


In [ ]:
# Python 3.11 must exist; no silent version substitution or kernel replacement.
PYTHON311 = shutil.which("python3.11") or (sys.executable if sys.version_info[:2] == (3, 11) else None)
try:
    assert PYTHON311, "CANONICAL ENVIRONMENT MISMATCH: Python 3.11 executable unavailable"
    actual = subprocess.check_output([PYTHON311, "-c", "import sys; print('%d.%d' % sys.version_info[:2])"], text=True).strip()
    assert actual == "3.11", "CANONICAL ENVIRONMENT MISMATCH: Python version"
    VENV = Path("/content") / (RUN_ID + "-venv")
    assert not VENV.exists(), "Existing venv; no automatic replacement"
    subprocess.run([PYTHON311, "-m", "venv", str(VENV)], check=True)
    PYTHON = str(VENV / "bin/python")
    with (SETUP / "dependency-install.log").open("x") as log:
        completed = subprocess.run([PYTHON, "-m", "pip", "install", "-r", "experiments/requirements-scalar-smoke.txt"], cwd=RUN_REPO, env=RUN_ENV, stdout=log, stderr=subprocess.STDOUT, timeout=1800)
    assert completed.returncode == 0, "CANONICAL ENVIRONMENT MISMATCH: pinned installation failed"
    with (SETUP / "pip-check.log").open("x") as log:
        subprocess.run([PYTHON, "-m", "pip", "check"], check=True, stdout=log, stderr=subprocess.STDOUT, timeout=120)
except Exception as exc:
    (SETUP / "failure.json").write_text(json.dumps({"stage": "dependencies", "error": str(exc), "verdict": "LIVE OBSERVER NOT VERIFIED"}, indent=2))
    print("CANONICAL ENVIRONMENT MISMATCH")
    raise
print("PINNED SETUP: PASS")


In [ ]:
# Required six-file measurement gate (>=230 passes, zero skips), plus
# tests/test_scalar_colab_smoke.py and tests/test_scalar_colab_git.py.
# pre_gpu_gate runs the exact commands and retains argv/results/log hashes.
source_gate(RUN_REPO, CODE_SHA, RUN_REL)
preflight_code = "from pathlib import Path; import sys; from experiments.scalar_colab_smoke import environment_gate,save,pre_gpu_gate; root=Path(sys.argv[1]); save(root/'environment.json', environment_gate()); pre_gpu_gate(root)"
with (SETUP / "preflight-driver.log").open("x") as log:
    preflight = subprocess.run([PYTHON, "-c", preflight_code, str(SETUP)], cwd=RUN_REPO, env=RUN_ENV, stdout=log, stderr=subprocess.STDOUT, timeout=900)
assert preflight.returncode == 0, "Pre-GPU tests failed; do not launch models. Preserve setup logs."
STACK = json.loads((SETUP / "environment.json").read_text())
print("GPU / STACK:", {k: STACK[k] for k in ("gpu_name", "driver", "python", "torch", "cuda", "vllm")})
print("PREFLIGHT:", json.loads((SETUP / "pre_gpu_tests.json").read_text()))
print("COLAB PREFLIGHT: PASS")


In [ ]:
# Only K=0/B1, then K=2/B1 if K=0 passes. No grid/performance cells.
source_gate(RUN_REPO, CODE_SHA, RUN_REL)
try:
    with (RUN_DIR / "orchestrator.log").open("x") as log:
        smoke = subprocess.run([PYTHON, "-m", "experiments.scalar_colab_smoke", "run", "--output", str(RUN_DIR / "smoke"), "--preflight-dir", str(SETUP)], cwd=RUN_REPO, env=RUN_ENV, stdout=log, stderr=subprocess.STDOUT, timeout=3000)
except subprocess.TimeoutExpired as exc:
    (RUN_DIR / "watchdog-failure.json").write_text(json.dumps({"stage": "smoke-watchdog", "error": str(exc), "verdict": "LIVE OBSERVER NOT VERIFIED"}, indent=2))
    (RUN_DIR / "smoke").mkdir(exist_ok=True)
# No assertion on smoke exit here: failed attempts must reach evidence handling.
source_gate(RUN_REPO, CODE_SHA, RUN_REL)
validation = subprocess.run([PYTHON, "-m", "experiments.scalar_colab_smoke", "validate", "--output", str(RUN_DIR / "smoke")], cwd=RUN_REPO, env=RUN_ENV, capture_output=True, text=True, timeout=120)
(RUN_DIR / "offline-validator.log").write_text(validation.stdout + validation.stderr)
REPORT = json.loads((RUN_DIR / "smoke/validation.json").read_text())
print("OFFLINE VALIDATOR:", REPORT["offline_validation"])
print("OBSERVER VERDICT:", REPORT["verdict"])


In [ ]:
# Copy immutable run -> EVIDENCE_REPO, fetch/equality gate, explicit staging,
# evidence-only commit, second race gate, normal push. No merge/rebase/force.
PUSH_RESULT = {"evidence_commit_sha": None, "evidence_push": "NOT ATTEMPTED"}
try:
    with github_auth(userdata.get("GITHUB_TOKEN")) as auth_env:
        PUSH_RESULT = push_evidence(RUN_REPO, EVIDENCE_REPO, CODE_SHA, RUN_REL, env=auth_env)
except Exception as exc:
    head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=EVIDENCE_REPO, text=True).strip()
    PUSH_RESULT = {"evidence_commit_sha": head if head != CODE_SHA else None, "evidence_push": "BLOCKED", "reason": str(exc)}
# Receipt stays outside both checkouts; no post-copy mutation of run evidence.
receipt = Path("/content") / (RUN_ID + "-push-receipt.json")
receipt.write_text(json.dumps(PUSH_RESULT, indent=2))
print("CODE_SHA:", CODE_SHA)
print("GPU / STACK:", {k: STACK[k] for k in ("gpu_name", "driver", "torch", "cuda", "vllm")})
print("SOURCE GATE:", source_gate(RUN_REPO, CODE_SHA, RUN_REL)["source_clean"])
print("PREFLIGHT: PASS")
for k in (0, 2):
    path = RUN_DIR / f"smoke/k{k}/cell.json"
    cell = json.loads(path.read_text()) if path.exists() else {"status": "NOT RUN"}
    print(f"K={k} / actual B=1 STRUCTURAL RESULT:", cell["status"])
print("OBSERVER VERDICT:", REPORT["verdict"])
print("OFFLINE VALIDATOR:", REPORT["offline_validation"])
print("EVIDENCE COMMIT SHA:", PUSH_RESULT["evidence_commit_sha"])
print("EVIDENCE PUSH:", PUSH_RESULT["evidence_push"])
print("PROFITABILITY: NOT EVALUATED — INSTRUMENTATION SMOKE ONLY")
print("FULL GRID: NOT RUN — AWAITING SMOKE REVIEW")
if PUSH_RESULT["evidence_push"] != "PASS":
    print("Preserved:", RUN_DIR, EVIDENCE_REPO)
    raise RuntimeError("Evidence push blocked; inspect auth/divergence and reconcile manually")
